# Day 4 · 실습 3. YOLO·SAM 결과 및 오류 사례 분석

**연계 이론:** Day 4 · 이론 4. CV 작업별 평가 지표  
**권장 운영:** 50분 / 저장된 결과 로딩·시각화 실행 목표 10분 이내  
**실습 유형:** B(조건 변경·비교) + C(시각화) + D(오류 분석 1~2개)

### 학습 목표
1. 앞선 YOLO 실습의 탐지 결과를 **간단한 수업용 정답 Bounding Box**와 비교합니다.
2. IoU 기준으로 TP·FP·FN을 판정하고 Confidence Threshold와의 관계를 설명합니다.
3. SAM의 서로 다른 Prompt 결과를 비교하되, **정답 Mask가 없으면 오류 판정이 불가능함**을 구분합니다.

> **실행 권장 순서:** YOLO Notebook(③교시) → SAM Notebook(⑤교시) → 이 Notebook(⑦교시). 앞선 결과 파일이 없다면 일부 셀은 **가상 데이터로 진행**하며, 그 경우 실제 모델의 오류로 해석하지 않습니다.

## 0. 실습 폴더와 저장된 결과 확인

이 Notebook은 **추가 모델 추론이나 가중치 다운로드를 하지 않습니다.** 실제 YOLO·SAM 결과가 있다면 우선 재사용하며, 없을 경우 별도로 표시한 교육용 가상 예제를 사용합니다.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# ZIP 안의 assets 폴더를 기준으로 작업 경로를 찾습니다.
def locate_root():
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        for candidate in (parent, parent / "cv_day4_practice_v1"):
            if (candidate / "assets" / "coffee.jpg").is_file():
                return candidate
    raise FileNotFoundError(
        "assets/coffee.jpg를 찾지 못했습니다. ZIP 전체를 압축 해제하고 "
        "cv_day4_practice_v1 폴더에서 Notebook을 실행하세요."
    )

ROOT = locate_root()
ASSET_DIR = ROOT / "assets"
OUTPUT_DIR = ROOT / "outputs"
WEIGHT_DIR = ROOT / "weights"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0])
print("실습 루트:", ROOT)
print("샘플 이미지:", [p.name for p in sorted(ASSET_DIR.glob("*.jpg"))])

In [ ]:
from matplotlib.patches import Rectangle
YOLO_FILE = OUTPUT_DIR / "yolo" / "yolo_detections.json"
SAM_DIR = OUTPUT_DIR / "sam"
GT_FILE = ASSET_DIR / "ground_truth_demo.json"

has_real_yolo = YOLO_FILE.is_file()
has_real_sam = all((SAM_DIR / f"mask_{key}.npy").is_file()
                   for key in ("positive", "positive_negative", "box"))
print("YOLO 저장 결과:", "있음" if has_real_yolo else "없음 — 명시된 가상 예제를 사용")
print("SAM 저장 Mask:", "있음" if has_real_sam else "없음 — 명시된 가상 예제를 사용")
with GT_FILE.open("r", encoding="utf-8") as f:
    gt_meta = json.load(f)
print("수업용 근사 GT 대상:", list(gt_meta["images"]))
print("주의: 근사 수업용 Annotation은 공인 데이터셋의 정식 Ground Truth가 아닙니다.")

## 1. YOLO 결과에서 TP·FP·FN 계산

실습용 사진에 **한 클래스만** 수동으로 근사 정답 Box를 표시했습니다. `coffee.jpg`는 `cup`, `chelsea.jpg`는 `cat`만 평가합니다. 이 파일은 **공식 라벨이 아니며**, 그림을 보고 근사한 수업용 Annotation이므로 평가 값도 연습용입니다. 다른 클래스 예측은 평가에서 제외합니다.

탐지 후보를 Confidence 높은 순으로 정렬하고, 동일 클래스이며 `IoU≥0.5`인 아직 매칭되지 않은 정답을 하나만 TP로 인정합니다. 매칭되지 않은 예측은 FP, 끝까지 매칭되지 않은 정답은 FN입니다.

In [ ]:
def box_iou_xyxy(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0., x2 - x1) * max(0., y2 - y1)
    aa = max(0., a[2] - a[0]) * max(0., a[3] - a[1])
    bb = max(0., b[2] - b[0]) * max(0., b[3] - b[1])
    union = aa + bb - inter
    return inter / union if union else 0.

def match_one_class(predictions, gt_boxes, target_class, iou_threshold=0.50):
    """하나의 클래스에 대해 일대일 매칭으로 TP/FP/FN 판정."""
    selected = sorted((p for p in predictions if p["class_name"] == target_class),
                      key=lambda p: p["confidence"], reverse=True)
    matched_gt = set()
    details = []
    for pred in selected:
        eligible = [(j, box_iou_xyxy(pred["xyxy"], gt)) for j, gt in enumerate(gt_boxes)
                    if j not in matched_gt]
        best = max(eligible, key=lambda item: item[1]) if eligible else None
        if best is not None and best[1] >= iou_threshold:
            matched_gt.add(best[0])
            details.append({**pred, "evaluation": "TP", "best_iou": best[1]})
        else:
            details.append({**pred, "evaluation": "FP", "best_iou": best[1] if best else 0.})
    tp = sum(d["evaluation"] == "TP" for d in details)
    fp, fn = len(details) - tp, len(gt_boxes) - tp
    precision = tp / (tp + fp) if (tp + fp) else None  # 0건이면 정의되지 않음
    recall = tp / (tp + fn) if (tp + fn) else None
    return {"tp": tp, "fp": fp, "fn": fn, "precision": precision,
            "recall": recall, "details": details, "matched_gt": sorted(matched_gt)}

if has_real_yolo:
    with YOLO_FILE.open("r", encoding="utf-8") as f:
        yolo_record = json.load(f)
    print("실제 실행 결과를 불러왔습니다:", YOLO_FILE)
else:
    # 아래는 모델 추론 결과가 절대 아닙니다. Notebook만 단독 실행할 때의 가상 교육 사례입니다.
    yolo_record = {"images": {
        "coffee.jpg": {"thresholds": {
            "0.15": [{"class_name":"cup", "confidence":0.93, "xyxy":[165,20,423,304]},
                     {"class_name":"cup", "confidence":0.19, "xyxy":[40,40,150,120]}],
            "0.30": [{"class_name":"cup", "confidence":0.93, "xyxy":[165,20,423,304]}],
            "0.60": [{"class_name":"cup", "confidence":0.93, "xyxy":[165,20,423,304]}],
        }},
        "chelsea.jpg": {"thresholds": {
            "0.15": [{"class_name":"cat", "confidence":0.75, "xyxy":[4,1,443,295]}],
            "0.30": [{"class_name":"cat", "confidence":0.75, "xyxy":[4,1,443,295]}],
            "0.60": [{"class_name":"cat", "confidence":0.75, "xyxy":[4,1,443,295]}],
        }}
    }}
    print("*** 교육용 가상 YOLO 예측값입니다. 실제 YOLO 결과가 아닙니다. ***")

In [ ]:
IOU_EVAL_THRESHOLD = 0.50   # [조건 변경] 평가 매칭의 기준 (NMS IoU 기준과 다름)
metrics_rows = []
analyses = {}
for image_name, gt_info in gt_meta["images"].items():
    target_class = gt_info["target_class"]
    gt_boxes = gt_info["boxes_xyxy"]
    record = yolo_record["images"].get(image_name, {})
    analyses[image_name] = {}
    for threshold_key, predictions in record.get("thresholds", {}).items():
        result = match_one_class(predictions, gt_boxes, target_class, IOU_EVAL_THRESHOLD)
        analyses[image_name][threshold_key] = result
        metrics_rows.append({"image": image_name, "class": target_class,
                             "conf": threshold_key, "TP": result["tp"],
                             "FP": result["fp"], "FN": result["fn"],
                             "precision": result["precision"], "recall": result["recall"]})
for row in metrics_rows:
    p = f'{row["precision"]:.3f}' if row["precision"] is not None else 'N/A'
    r = f'{row["recall"]:.3f}' if row["recall"] is not None else 'N/A'
    print(f'{row["image"]:12} class={row["class"]:4} conf={row["conf"]:>4} '
          f'TP={row["TP"]} FP={row["FP"]} FN={row["FN"]} P={p} R={r}')
print("N/A: TP+FP가 0이면 Precision 분모가 0이므로 여기서는 정의하지 않습니다.")

## 2. TP·FP·FN의 위치 확인

점선 Box는 사람이 표시한 **근사 GT**, 실선은 모델의 예측입니다. 녹색 실선=TP, 붉은 실선=FP, 매칭되지 않은 GT=FN입니다. 주의: 이 판정은 수업용 GT가 표시된 **한 클래스**만을 대상으로 합니다.

In [ ]:
image_name = "coffee.jpg"  # [조건 변경] "chelsea.jpg"로 바꿔보기
threshold_key = "0.30"  # [조건 변경] YOLO Notebook에서 저장된 Threshold 키 중 하나
info = gt_meta["images"][image_name]
analysis = analyses.get(image_name, {}).get(threshold_key)
if analysis is None:
    raise KeyError(f"저장된 결과에 {image_name} conf={threshold_key}가 없습니다. 실제 저장한 값을 확인하세요.")

img = np.asarray(Image.open(ASSET_DIR / image_name).convert("RGB"))
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(img)
for j, (x1,y1,x2,y2) in enumerate(info["boxes_xyxy"]):
    is_fn = j not in analysis["matched_gt"]
    ax.add_patch(Rectangle((x1,y1), x2-x1, y2-y1, fill=False,
                           edgecolor="orange" if is_fn else "yellow", linewidth=2.5, linestyle="--"))
    ax.text(x1, max(0,y1-7), "GT: FN" if is_fn else "GT: matched", color="yellow", fontsize=10,
            bbox={"facecolor":"black", "alpha":0.6})
for pred in analysis["details"]:
    x1,y1,x2,y2 = pred["xyxy"]
    color = "lime" if pred["evaluation"] == "TP" else "red"
    ax.add_patch(Rectangle((x1,y1), x2-x1, y2-y1, fill=False, edgecolor=color, linewidth=2))
    ax.text(x1, min(img.shape[0]-4, y2+12), f'{pred["evaluation"]} {pred["confidence"]:.2f}',
            color=color, fontsize=10, bbox={"facecolor":"black", "alpha":0.65})
ax.axis("off")
ax.set_title(f'{image_name} / class={info["target_class"]} / conf={threshold_key} '
             + ("[model output]" if has_real_yolo else "[SYNTHETIC predictions]"))
fig.tight_layout()
ERR_OUT = OUTPUT_DIR / "analysis"
ERR_OUT.mkdir(parents=True, exist_ok=True)
fig.savefig(ERR_OUT / f"boxes_{image_name[:-4]}_{threshold_key.replace('.', '-')}.png", dpi=140, bbox_inches="tight")
plt.show()

## 3. SAM Prompt별 Mask 차이 확인

다음 두 Mask를 비교할 때 한 가지를 구분합니다. **일치도가 낮다**는 사실은 알 수 있지만, 정답 Mask가 없다면 어느 쪽이 옳은지 판정할 수 없습니다. 앞선 SAM 실습 결과가 존재하면 그대로 읽습니다.

In [ ]:
def calculate_mask_agreement(a, b):
    a, b = np.asarray(a, bool), np.asarray(b, bool)
    inter = int((a & b).sum()); union = int((a | b).sum())
    denom = int(a.sum() + b.sum())
    return (inter/union if union else 1.0, 2*inter/denom if denom else 1.0)

if has_real_sam:
    mask_a = np.load(SAM_DIR / "mask_positive.npy").astype(bool)
    mask_b = np.load(SAM_DIR / "mask_positive_negative.npy").astype(bool)
    sam_image = np.asarray(Image.open(ASSET_DIR / "coffee.jpg").convert("RGB"))
    example_label = "SAM 2.1 saved predictions"
else:
    # 실제 SAM 결과 아님: 실습 이해를 위한 두 개의 가상 마스크
    sam_image = np.asarray(Image.open(ASSET_DIR / "coffee.jpg").convert("RGB"))
    h, w = sam_image.shape[:2]
    yy, xx = np.ogrid[:h, :w]
    mask_a = ((xx - 265)/125)**2 + ((yy - 185)/155)**2 <= 1
    mask_b = ((xx - 275)/110)**2 + ((yy - 190)/135)**2 <= 1
    example_label = "SYNTHETIC example, not SAM output"

agreement_iou, agreement_dice = calculate_mask_agreement(mask_a, mask_b)
only_a, only_b = mask_a & ~mask_b, mask_b & ~mask_a
print(example_label)
print(f"Prompt 간 일치도 IoU={agreement_iou:.3f}, Dice={agreement_dice:.3f}")
print(f"A에만 포함: {int(only_a.sum()):,}px | B에만 포함: {int(only_b.sum()):,}px")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, mask, title in [(axes[0], mask_a, "Positive"),
                        (axes[1], mask_b, "Positive + Negative")]:
    viz = sam_image.astype(float).copy()
    viz[mask] = viz[mask]*.6 + np.array([40,220,100])*.4
    ax.imshow(viz.astype(np.uint8)); ax.axis("off"); ax.set_title(title)
colors = np.zeros((*mask_a.shape, 3), dtype=np.uint8)
colors[mask_a & mask_b] = [70, 185, 95]   # 공통 영역
colors[only_a] = [255, 110, 80]          # A만
colors[only_b] = [75, 145, 250]          # B만
axes[2].imshow(colors); axes[2].axis("off")
axes[2].set_title("Green=both / Orange=A only / Blue=B only")
fig.suptitle(example_label)
fig.tight_layout(); fig.savefig(ERR_OUT / "sam_mask_change.png", dpi=140, bbox_inches="tight"); plt.show()
print("정답 Mask가 없기 때문에 위 수치는 실제 모델 정확도가 아닙니다.")

## 4. 정답 Mask가 있을 때의 분할 오류 계산 (가상 데이터)

마지막 예제는 **정답이 알려진 단순 도형**을 사용하여 어떤 픽셀이 FP/FN인지 직접 확인합니다. 실제 SAM 출력은 아니며, Mask IoU·Dice 계산을 복습하기 위한 별도 사례입니다.

In [ ]:
size = 128
yy, xx = np.ogrid[:size, :size]
gt = ((xx-62)**2/(37**2) + (yy-65)**2/(30**2)) <= 1
pred = ((xx-72)**2/(37**2) + (yy-60)**2/(30**2)) <= 1
pixel_tp = int((gt & pred).sum())
pixel_fp = int((~gt & pred).sum())
pixel_fn = int((gt & ~pred).sum())
mask_iou = pixel_tp / (pixel_tp + pixel_fp + pixel_fn)
mask_dice = 2*pixel_tp / (2*pixel_tp + pixel_fp + pixel_fn)
print(f"TP pixels={pixel_tp:,}, FP pixels={pixel_fp:,}, FN pixels={pixel_fn:,}")
print(f"Mask IoU={mask_iou:.3f}, Dice={mask_dice:.3f}")

colored = np.zeros((size,size,3),dtype=np.uint8)
colored[gt & pred] = [70, 175, 95]       # TP (녹색)
colored[~gt & pred] = [245, 115, 75]     # FP (주황)
colored[gt & ~pred] = [60, 130, 245]     # FN (파랑)
fig, ax = plt.subplots(figsize=(6,6))
ax.imshow(colored); ax.axis("off")
ax.set_title(f"SYNTHETIC mask errors | IoU={mask_iou:.3f}, Dice={mask_dice:.3f}")
fig.tight_layout(); fig.savefig(ERR_OUT / "synthetic_mask_fp_fn.png", dpi=140, bbox_inches="tight"); plt.show()

## 5. 결과를 저장하고 해석하기

- 오류 사례 ①: **탐지 결과와 근사 GT를 비교**하여 TP·FP·FN의 이유를 설명하세요. Confidence와 **평가용 IoU Threshold**는 서로 다른 기준입니다.
- 오류 사례 ②: **SAM Prompt 변경** 후 Mask가 달라진 픽셀을 살펴보세요. 이때 정확도라는 표현을 사용하려면 정답 Mask가 필요합니다.
- 학생 스스로 답해 보기: `Confidence 0.15`와 `0.60`에서 `cup` 클래스의 TP·FP·FN은 어떻게 다른가요? 분할 Prompt를 바꾸었을 때 실제로 제외된 영역은 어디인가요?

**이번 수업 정리:** Classification은 클래스 결과, Detection은 클래스+위치, Segmentation은 픽셀 영역을 평가합니다. GT가 없으면 시각적인 관찰과 모델 평가를 구분해야 합니다.

In [ ]:
import csv
with (ERR_OUT / "detection_error_metrics.csv").open("w", encoding="utf-8-sig", newline="") as f:
    keys = ["image", "class", "conf", "TP", "FP", "FN", "precision", "recall"]
    writer = csv.DictWriter(f, fieldnames=keys)
    writer.writeheader(); writer.writerows(metrics_rows)
notes = {
    "yolo_result_type": "real model outputs" if has_real_yolo else "synthetic educational example",
    "sam_result_type": "real model outputs" if has_real_sam else "synthetic educational example",
    "gt_type": "manually approximated demonstration bounding boxes, NOT official dataset GT",
    "mask_agreement_iou": agreement_iou,
    "mask_agreement_dice": agreement_dice,
    "synthetic_gt_mask_iou": mask_iou,
    "synthetic_gt_mask_dice": mask_dice,
}
with (ERR_OUT / "analysis_notes.json").open("w", encoding="utf-8") as f:
    json.dump(notes, f, ensure_ascii=False, indent=2)
print("⑦교시 결과 저장:", ERR_OUT)
print("완료. outputs/ 폴더에서 비교 이미지·CSV·JSON을 확인하세요.")